# Arabic Sentiment Analysis — Fine-tuning AraBERT

**Project:** Arabic Sentiment Analysis  
**Model:** [aubmindlab/bert-base-arabertv02](https://huggingface.co/aubmindlab/bert-base-arabertv02)  
**Dataset:** Arabic 100K Reviews (Positive / Neutral / Negative)  
**Task:** 3-class sentiment classification  

---

### Pipeline Overview
1. Load & validate cleaned dataset  
2. Encode labels and split data  
3. Tokenize with AraBERT tokenizer  
4. Fine-tune with HuggingFace Trainer  
5. Evaluate with Macro F1 and classification report  
6. Save model for deployment

## 1. Imports

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## 2. Load & Validate Dataset

The dataset is the **Arabic 100K Reviews** corpus containing hotel, book, and product reviews.  
Labels: `Positive`, `Neutral` (originally *Mixed*), `Negative` — balanced at ~33K each.

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/mazinwael/cleaned-data-csv/cleaned_data.csv')
df = df.dropna(subset=['Cleaned_Text', 'label']).copy()
df['Cleaned_Text'] = df['Cleaned_Text'].astype(str).str.strip()
df['label']        = df['label'].astype(str).str.strip()
df = df[df['Cleaned_Text'] != ''].copy()

print(f"Dataset shape : {df.shape}")
print()
print("Label distribution:")
print(df['label'].value_counts())
df.head()

## 3. Label Encoding

Map string labels to integer IDs required by the model.

In [ ]:
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

df['label_id'] = df['label'].map(label2id)

print("label_id distribution:")
print(df['label_id'].value_counts())

## 4. Train / Validation Split

- **80% train** / **20% validation**  
- `stratify=label_id` ensures balanced class distribution in both splits  
- `random_state=42` for reproducibility

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    df['Cleaned_Text'].tolist(),
    df['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label_id'],
    shuffle=True,
)

print(f"Train size      : {len(X_train):,}")
print(f"Validation size : {len(X_val):,}")

## 5. Tokenizer

AraBERT uses a custom Arabic tokenizer that handles:
- Arabic morphology (root-based word structure)  
- Diacritics normalization  
- Subword tokenization (WordPiece)  

`max_length=128` covers ~95% of reviews without truncation.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")


## 6. PyTorch Dataset

Wraps tokenized inputs in a `Dataset` object compatible with HuggingFace `Trainer`.

In [ ]:
class ArabicSentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=128,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


train_dataset = ArabicSentimentDataset(X_train, y_train)
val_dataset   = ArabicSentimentDataset(X_val,   y_val)

print(f"Train dataset size : {len(train_dataset):,}")
print(f"Val dataset size   : {len(val_dataset):,}")

## 7. Load Pre-trained Model

`AutoModelForSequenceClassification` adds a **classification head** on top of AraBERT.

> **Note on warnings:**  
> - *Missing keys* — the classification head is new and randomly initialized (expected).  
> - *Unexpected keys* — MLM/NSP heads from pre-training are discarded (expected).  
> This is standard transfer learning behavior.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv02",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print("Model loaded successfully.")

## 8. Fine-tuning

### Key hyperparameters

| Parameter | Value | Reason |
|-----------|-------|--------|
| `learning_rate` | 1e-5 | Small LR prevents overwriting pre-trained weights |
| `num_train_epochs` | 2 | Sufficient — more epochs cause overfitting |
| `warmup_steps` | 10% of steps | Gradual ramp-up stabilizes early training |
| `weight_decay` | 0.01 | L2 regularization reduces overfitting |
| `metric_for_best_model` | macro_f1 | Balanced metric for 3 equal classes |

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "macro_f1" : f1_score(labels, predictions, average="macro"),
    }

epochs          = 2
train_batch_size = 16
steps_per_epoch  = max(1, len(train_dataset) // train_batch_size)
warmup_steps     = int(steps_per_epoch * epochs * 0.1)

training_args = TrainingArguments(
    output_dir               = "./results",
    num_train_epochs         = epochs,
    per_device_train_batch_size = train_batch_size,
    per_device_eval_batch_size  = 16,
    eval_strategy            = "epoch",
    save_strategy            = "epoch",
    load_best_model_at_end   = True,
    metric_for_best_model    = "macro_f1",
    greater_is_better        = True,
    learning_rate            = 1e-5,
    warmup_steps             = warmup_steps,
    weight_decay             = 0.01,
    logging_steps            = 50,
    save_total_limit         = 2,
    report_to                = "none",
    fp16                     = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    compute_metrics = compute_metrics,
)

train_result = trainer.train()
print(train_result)

## 9. Evaluation

### Results

| Model | Accuracy | Macro F1 |
|-------|----------|----------|
| TF-IDF + Logistic Regression (baseline) | 66% | 0.66 |
| **AraBERT fine-tuned** | **74%** | **0.74** |

AraBERT improves over the baseline by **+8%** on Macro F1.

In [ ]:
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(
    y_val,
    y_pred,
    target_names=[id2label[i] for i in sorted(id2label)],
))

## 10. Save Model

Save the fine-tuned model and tokenizer for deployment via FastAPI.

In [ ]:
trainer.save_model("./arabert-sentiment")
tokenizer.save_pretrained("./arabert-sentiment")
print("Model and tokenizer saved to ./arabert-sentiment")

## 11. Limitations & Future Work

| Limitation | Description |
|------------|-------------|
| Dialect coverage | Model performs best on Modern Standard Arabic (MSA); Egyptian/Levantine dialects may be misclassified |
| Negation handling | Phrases like *مش وحش* (means Positive) can be misclassified due to the word *وحش* |
| Domain bias | Training data is reviews (hotels, books, products) — may not generalize to other domains |

### Future improvements
- Add Egyptian dialect dataset (e.g. EJAD) and retrain  
- Experiment with `bert-base-arabic-camelbert-da-sentiment` for better dialect support  
- Try 3–5 epochs with early stopping on a larger GPU